In [7]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# --- Data Import ---
def import_data(csv_file):
    df = pd.read_csv(csv_file)
    patient_ids = df.iloc[:, 0].astype(str).str.strip()
    features = df.iloc[:, 1:208]  # Adjust if needed
    feats = []
    unique_ids = patient_ids.unique()
    for pid in unique_ids:
        patient_data = features[patient_ids == pid].values
        feats.append(patient_data)
    return feats, unique_ids

def import_labels(csv_file, label_column_name):
    df = pd.read_csv(csv_file)
    patient_ids = df.iloc[:, 0].astype(str).str.strip()
    labels_df = df[[df.columns[0], label_column_name]].copy()
    labels_df[df.columns[0]] = labels_df[df.columns[0]].astype(str).str.strip()
    labels_df = labels_df.drop_duplicates(subset=df.columns[0]).set_index(df.columns[0])
    unique_ids = patient_ids.unique()
    labels = labels_df.loc[unique_ids][label_column_name].values
    return labels.astype(np.int64)

# --- Standardization ---
def standardize_features(feats):
    mean_record = [np.mean(case, axis=0) for case in feats]
    std_record = [np.std(case, axis=0) for case in feats]
    feature_mean = np.nanmean(mean_record, axis=0)
    feature_std = np.nanstd(std_record, axis=0)
    feature_std = np.nan_to_num(feature_std)
    feature_mean[feature_mean == np.inf] = 0

    standardized_feats = []
    for patient in feats:
        patient = np.array(patient)
        patient -= feature_mean
        patient /= feature_std
        patient = np.nan_to_num(patient, nan=0.0, posinf=0.0, neginf=0.0)
        standardized_feats.append(patient)
    return standardized_feats

# --- Aggregation ---
def aggregate_patient_features(feats, method="mean"):
    agg_feats = []
    for patient in feats:
        if method == "mean":
            agg_feats.append(np.mean(patient, axis=0))
        elif method == "max":
            agg_feats.append(np.max(patient, axis=0))
        elif method == "min":
            agg_feats.append(np.min(patient, axis=0))
        else:
            raise ValueError("Unsupported aggregation method.")
    return np.array(agg_feats)

# --- Save Results ---
def save_results_csv(true_labels, pred_labels, pred_probs, filename):
    df = pd.DataFrame({
        "TrueLabel": true_labels,
        "PredLabel": pred_labels,
        "PredProb": pred_probs
    })
    df.to_csv(filename, index=False)
    print(f"Saved predictions to {filename}")

def plot_and_save_confusion_matrix(true_labels, pred_labels, filename):
    cm = confusion_matrix(true_labels, pred_labels)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1], yticklabels=[0,1])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(filename)
    plt.close()
    print(f"Saved confusion matrix to {filename}")

def plot_and_save_roc_curve(true_labels, pred_probs, filename):
    fpr, tpr, _ = roc_curve(true_labels, pred_probs)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label="ROC Curve")
    plt.plot([0,1], [0,1], "k--", label="Random")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.savefig(filename)
    plt.close()
    print(f"Saved ROC curve to {filename}")



In [8]:
def cross_validate_xgboost(csv_file, label_column, folds=10, agg_method="mean", save_directory="./", num_est=300, seed=None):
    print("Loading data...")
    feats, unique_ids = import_data(csv_file)
    labels = import_labels(csv_file, label_column)

    print(f"Total patients: {len(labels)}")
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)

    all_true, all_pred, all_prob = [], [], []
    best_params_list = []

    os.makedirs(save_directory, exist_ok=True)

    param_grid = {
        'max_depth': [3, 4, 5, 6],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'n_estimators': [100, 200, 300],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
    }

    for fold, (train_idx, test_idx) in enumerate(skf.split(feats, labels), 1):
        print(f"\nFold {fold}/{folds}")
        train_feats = [feats[i] for i in train_idx]
        test_feats = [feats[i] for i in test_idx]
        y_train, y_test = labels[train_idx], labels[test_idx]

        # --- Flatten all train embeddings and standardize ---
        all_train_embeddings = np.vstack(train_feats)
        scaler = StandardScaler()
        all_train_embeddings = scaler.fit_transform(all_train_embeddings)

        # --- Apply scaler to per-patient train/test features ---
        # Redo train_feats and test_feats as transformed individual arrays
        train_feats_scaled = []
        start = 0
        for f in train_feats:
            n = len(f)
            train_feats_scaled.append(all_train_embeddings[start:start + n])
            start += n

        # Apply same scaler to test features
        test_feats_scaled = [scaler.transform(p) for p in test_feats]

        # --- PCA Fit Only on Standardized Train ---
        pca = PCA(n_components=0.99)
        pca.fit(np.vstack(train_feats_scaled))
        print(f"PCA reduced embedding dim to {pca.n_components_} for fold {fold}")

        train_feats_pca = [pca.transform(p) for p in train_feats_scaled]
        test_feats_pca = [pca.transform(p) for p in test_feats_scaled]

        # --- Aggregation ---
        X_train = aggregate_patient_features(train_feats_pca, method=agg_method)
        X_test = aggregate_patient_features(test_feats_pca, method=agg_method)

        # --- Standardize final aggregated vectors (optional but recommended) ---
        final_scaler = StandardScaler()
        X_train = final_scaler.fit_transform(X_train)
        X_test = final_scaler.transform(X_test)

        X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
        X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

        base_model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            scale_pos_weight=(sum(y_train == 0) / sum(y_train == 1)),
            random_state=42
        )

        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            scoring='roc_auc',
            cv=3,
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train, y_train)
        best_params = grid_search.best_params_
        best_params_list.append(best_params)
        print(f"Best params for fold {fold}: {best_params}")

        best_model = XGBClassifier(
            **best_params,
            objective='binary:logistic',
            eval_metric='logloss',
            scale_pos_weight=(sum(y_train == 0) / sum(y_train == 1)),
            verbosity=0,
            random_state=42
        )

        best_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

        prob = best_model.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.35).astype(int)

        all_true.extend(y_test)
        all_pred.extend(pred)
        all_prob.extend(prob)

        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)
        mcc = matthews_corrcoef(y_test, pred)
        print(f"Fold {fold} Acc: {acc:.4f} | AUC: {auc:.4f} | MCC: {mcc:.4f}")

    # --- Final Results ---
    report = classification_report(all_true, all_pred, digits=4)
    acc = accuracy_score(all_true, all_pred)
    auc = roc_auc_score(all_true, all_prob)
    mcc = matthews_corrcoef(all_true, all_pred)
    tn, fp, fn, tp = confusion_matrix(all_true, all_pred).ravel()
    specificity = tn / (tn + fp)

    print("\nClassification Report (Aggregated):")
    print(report)
    print(f"Overall Accuracy: {acc:.4f}")
    print(f"Overall AUC: {auc:.4f}")
    print(f"Overall MCC: {mcc:.4f}")
    print(f"Overall Specificity: {specificity:.4f}")

    results_path = os.path.join(save_directory, f"{os.path.basename(save_directory)}_results.txt")
    with open(results_path, "w") as f:
        f.write("Classification Report (Aggregated):\n")
        f.write(report)
        f.write(f"\nOverall Accuracy: {acc:.4f}\n")
        f.write(f"Overall AUC: {auc:.4f}\n")
        f.write(f"Overall MCC: {mcc:.4f}\n")
        f.write(f"Overall Specificity: {specificity:.4f}\n")
        f.write("\nBest Params Per Fold:\n")
        for i, params in enumerate(best_params_list, 1):
            f.write(f"Fold {i}: {params}\n")

    save_results_csv(all_true, all_pred, all_prob, os.path.join(save_directory, f"{os.path.basename(save_directory)}_predictions.csv"))
    plot_and_save_confusion_matrix(all_true, all_pred, os.path.join(save_directory, f"{os.path.basename(save_directory)}_confusion_matrix.png"))
    plot_and_save_roc_curve(all_true, all_prob, os.path.join(save_directory, f"{os.path.basename(save_directory)}_roc_curve.png"))


In [9]:
# --- Run ---
csv_file = "/blue/pinaki.sarder/narra.sr/5_DN_SKorea/data/combined_tubular_features_with_outcomes.csv"
label_column = "esrd"
cross_validate_xgboost(csv_file, label_column, agg_method="mean", save_directory='./xgb_pca_grid_0.99_seed_42', num_est=300, seed = 42)

Loading data...
Total patients: 86

Fold 1/10
PCA reduced embedding dim to 84 for fold 1
Best params for fold 1: {'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Fold 1 Acc: 0.8889 | AUC: 0.8571 | MCC: 0.7559

Fold 2/10
PCA reduced embedding dim to 84 for fold 2
Best params for fold 2: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.6}
Fold 2 Acc: 0.8889 | AUC: 0.7143 | MCC: 0.6614

Fold 3/10
PCA reduced embedding dim to 84 for fold 3
Best params for fold 3: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100, 'subsample': 0.6}
Fold 3 Acc: 0.8889 | AUC: 0.9286 | MCC: 0.7559

Fold 4/10
PCA reduced embedding dim to 84 for fold 4
Best params for fold 4: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.6}
Fold 4 Acc: 0.6667 | AUC: 0.8571 | MCC: 0.4781

Fold 5/10
PCA reduced embedding dim to 84 for fold